This notebook makes sure that the whole Fisher pipeline runs smoothly and prepares the 3d plotting code needed for our money plot.

In [1]:
from pathlib import Path
import jax.numpy as np

from gwfast.gwfastGlobals import detectors as det_dict, detPath
import gwfast.waveforms as waveforms
from gwfast.detector import Detector
from gwfast.signals import AGNLensedGWSignal
import gwfast.network as network
from gwfast.fisherTools import reduce_Fisher_matrix, CovMatr, plot_corners

LSC Algorithm Library (LAL) is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH
TEOBResumS is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH


## Test Fisher (with 3d input)

In [2]:
# Set up detectors
H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
L1 = Detector('L1', **det_dict['L1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
V1 = Detector('V1', **det_dict['Virgo'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/avirgo_O5low_NEW.txt')

wf_model = waveforms.IMRPhenomD()

H1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=H1, fmin=10)
L1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=L1, fmin=10)
V1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=V1, fmin=10)
HLV_AGN = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN, 'V1': V1_AGN})

Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1


In [3]:
# Define a benchmark event
n = 2
shape = (n, n, n)
events = {
    'Mc':np.full(shape, 30), 'eta':np.full(shape, 0.24), 
    'chi1z':np.full(shape, 0.3), 'chi2z':np.full(shape, 0.5), 
    'tcoal':np.full(shape, 0), 'phase':np.full(shape, 2), 
    'R_orbit':np.full(shape, 100), 'M_lz':np.full(shape, 1e4), 'src_pos':np.full(shape, 0.5),
    'iota':np.full(shape, 0.99*np.pi/2), 'psi':np.full(shape, 4), 
    'dL':np.full(shape, 0.8), 'theta':np.full(shape, 1.87), 'phi':np.full(shape, 2.66), 
}
events = {key: val.astype(np.float64) for key, val in events.items()}

# Sample parameter space
R_orbit_array = np.geomspace(20, 5000, n)
src_pos_array = np.linspace(0.01, 0.99, n)
dL_array = np.geomspace(0.1, 10, n)

# Make grid
R_orbit_mesh, src_pos_mesh, dL_mesh = np.meshgrid(R_orbit_array, src_pos_array, dL_array, indexing='xy')
events['R_orbit'] = R_orbit_mesh
events['src_pos'] = src_pos_mesh
events['dL'] = dL_mesh

In [4]:
import importlib
import gwfast.fisherTools
importlib.reload(gwfast.fisherTools)
from gwfast.fisherTools import reduce_Fisher_matrix, CovMatr, plot_corners

In [5]:
fisher_AGN = HLV_AGN.FisherMatr(events, res=1000)
keys = list(events.keys()).copy()
reduced_fisher_AGN, _ = reduce_Fisher_matrix(fisher_AGN, keys=keys)
reduced_cov_AGN, ie = CovMatr(reduced_fisher_AGN)

Computing Fisher for H1...
Computing Fisher for L1...
Computing Fisher for V1...
Done.


## Test 3d plotting isosurface

In [9]:
import numpy as onp
from skimage.measure import marching_cubes
from scipy.interpolate import interp1d
import pyvista as pv

In [10]:
n = 50
R_orbit_array = onp.geomspace(20, 5000, n)
src_pos_array = onp.linspace(0.01, 0.99, n)
dL_array = onp.geomspace(0.1, 10, n) # Gpc

R_orbit_mesh, src_pos_mesh, dL_mesh = onp.meshgrid(R_orbit_array, src_pos_array, dL_array, indexing='xy')
M_lens = 1e5 # M_sun
M_lens_mesh = onp.full(dL_mesh.shape, M_lens)
delta_mesh = 4.785e-20 * M_lens_mesh / (dL_mesh * 10)
theta_E = delta_mesh * onp.sqrt((2 * R_orbit_mesh * onp.sqrt(1 - src_pos_mesh ** 2)) / 
                                         (1 + delta_mesh * R_orbit_mesh * onp.sqrt(1 - src_pos_mesh ** 2)))
theta_E_in_arcsec = theta_E / onp.pi * 180 * 3600
print(onp.max(theta_E_in_arcsec), onp.min(theta_E_in_arcsec), onp.average(theta_E_in_arcsec), onp.median(theta_E_in_arcsec))

9.869524225277581e-08 2.3445005559330573e-11 6.545151124762322e-09 2.133393460611529e-09


In [57]:
def plot_isosurface(plotter, x, y, z, volume, level, color, opacity=0.8):
    verts, faces, normals, values_on_surface = marching_cubes(volume, level=level)
    verts[:, 0] /= verts[:, 0].max()
    verts[:, 1] /= verts[:, 1].max()
    verts[:, 2] /= verts[:, 2].max()

    faces_pv = onp.hstack([[3, *face] for face in faces])
    mesh = pv.PolyData(verts, faces_pv)
    plotter.add_mesh(mesh, color=color, opacity=opacity, show_edges=True)

In [58]:
theta_E_levels = [1e-10, 1e-9, 1e-8]
colors = ['firebrick', 'tomato', 'darkorange']

plotter = pv.Plotter()
for i in range(3):
    plot_isosurface(plotter, R_orbit_array, src_pos_array, dL_array, \
                    theta_E_in_arcsec, theta_E_levels[i], colors[i])

n_labels = 5
interval = 1 / (n_labels - 1)
ticks = [i * interval for i in range(n_labels)]
input_idx_interval = n / (n_labels - 1)

x_ticks = ticks
x_labels = [int(round(R_orbit_array[int(i * (input_idx_interval - 0.25))], 0)) for i in range(n_labels)]
y_ticks = ticks
y_labels = [src_pos_array[int(i * (input_idx_interval - 0.25))] for i in range(n_labels)]
z_ticks = ticks
z_labels = [round(dL_array[int(i * (input_idx_interval - 0.25))], 2) for i in range(n_labels)]

x_points = onp.array([[x, 0, 0] for x in x_ticks])
y_points = onp.array([[0, y, 0] for y in y_ticks])
z_points = onp.array([[0, 0, z] for z in z_ticks])

plotter.add_point_labels(x_points, x_labels, font_size=12, point_size=0)
plotter.add_point_labels(y_points, y_labels, font_size=12, point_size=0)
plotter.add_point_labels(z_points, z_labels, font_size=12, point_size=0)

plotter.show_bounds(
    grid='front',
    location='outer',
    all_edges=True,
    xlabel='R_orbit',
    ylabel='y',
    zlabel='dL'
)
plotter.show()

/Users/yuranzhang/gwfast_AGN/gwfast_moddd/venv/lib/python3.10/site-packages/pyvista/plotting/renderer.py:1846: PyVistaDeprecationWarning: `xlabel` is deprecated. Use `xtitle` instead.
  warnings.warn(
/Users/yuranzhang/gwfast_AGN/gwfast_moddd/venv/lib/python3.10/site-packages/pyvista/plotting/renderer.py:1852: PyVistaDeprecationWarning: `ylabel` is deprecated. Use `ytitle` instead.
  warnings.warn(
/Users/yuranzhang/gwfast_AGN/gwfast_moddd/venv/lib/python3.10/site-packages/pyvista/plotting/renderer.py:1858: PyVistaDeprecationWarning: `zlabel` is deprecated. Use `ztitle` instead.
  warnings.warn(


Widget(value='<iframe src="http://localhost:55968/index.html?ui=P_0x15902aaa0_24&reconnect=auto" class="pyvist…